# DeBERTa-v3-base Model Training for PCL Classification

This notebook implements the proposed approach for Exercise 4:
- **Model**: DeBERTa-v3-base (microsoft/deberta-v3-base)
- **Training Strategy**: 4-5 epochs with early stopping
- **Class Weighting**: Inverse frequency weighting (1.53:1)
- **Max Sequence Length**: 192 tokens
- **Batch Size**: 32
- **Learning Rate**: 1e-5 with warmup and linear decay

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install sentencepiece protobuf transformers datasets torch scikit-learn pandas numpy matplotlib seaborn tqdm accelerate nltk -q

## 2. Import Libraries

In [1]:
# Set HuggingFace cache directory
import os
os.environ['HF_HOME'] = '/data/ks2222/huggingface-cache'
os.environ['TRANSFORMERS_CACHE'] = '/data/ks2222/huggingface-cache'
os.environ['HF_DATASETS_CACHE'] = '/data/ks2222/huggingface-cache'

# Now import everything else
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DebertaV2Tokenizer,
    DebertaV2ForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"HuggingFace cache: {os.environ['HF_HOME']}")

/vol/bitbucket/ks2222/NLP_IndividualCW/BestModel/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA GeForce RTX 4080
HuggingFace cache: /data/ks2222/huggingface-cache


## 3. Load and Prepare Data

In [2]:
# Load the PCL dataset
url = 'https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv'

# Read with proper error handling
raw_df = pd.read_csv(url, sep='\t', header=None, on_bad_lines='skip', encoding='utf-8', engine='python')

# Find header row containing 'par_id'
header_candidates = raw_df.index[
    raw_df.apply(lambda r: r.astype(str).str.strip().str.lower().eq('par_id').any(), axis=1)
]

if len(header_candidates) > 0:
    header_row = int(header_candidates[0])
    df = pd.read_csv(
        url,
        sep='\t',
        skiprows=header_row,
        on_bad_lines='skip',
        encoding='utf-8',
        engine='python'
    )
else:
    # Fallback: known schema
    df = pd.read_csv(
        url,
        sep='\t',
        skiprows=4,
        names=['par_id', 'art_id', 'keyword', 'country', 'text', 'label'],
        on_bad_lines='skip',
        encoding='utf-8',
        engine='python'
    )

print(f"Loaded main dataset with {len(df)} samples")
print(f"Columns: {df.columns.tolist()}")

Loaded main dataset with 10469 samples
Columns: ['par_id', 'art_id', 'keyword', 'country', 'text', 'label']


In [3]:
# Load train/dev splits
train_split_url = 'https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv'
dev_split_url = 'https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv'

train_split = pd.read_csv(train_split_url)
dev_split = pd.read_csv(dev_split_url)

# Merge with main dataset
train_df = df.merge(train_split[['par_id']], on='par_id', how='inner')
dev_df = df.merge(dev_split[['par_id']], on='par_id', how='inner')

# Convert labels to numeric
train_df['label'] = pd.to_numeric(train_df['label'], errors='coerce')
dev_df['label'] = pd.to_numeric(dev_df['label'], errors='coerce')

# Remove any rows with NaN labels
train_df = train_df.dropna(subset=['label'])
dev_df = dev_df.dropna(subset=['label'])

# Convert labels to integers
train_df['label'] = train_df['label'].astype(int)
dev_df['label'] = dev_df['label'].astype(int)

# For binary classification, ensure we only have labels 0 and 1
# Filter out any other labels
train_df = train_df[train_df['label'].isin([0, 1])]
dev_df = dev_df[dev_df['label'].isin([0, 1])]

print(f"\nTrain set: {len(train_df)} samples")
print(f"Dev set: {len(dev_df)} samples")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts().sort_index())
print(f"\nDev label distribution:")
print(dev_df['label'].value_counts().sort_index())


Train set: 7581 samples
Dev set: 1895 samples

Train label distribution:
label
0    6825
1     756
Name: count, dtype: int64

Dev label distribution:
label
0    1704
1     191
Name: count, dtype: int64


In [4]:
# Calculate class weights (inverse frequency)
class_counts = train_df['label'].value_counts().sort_index()
total_samples = len(train_df)

# Ensure we have exactly 2 classes (0 and 1)
assert len(class_counts) == 2, f"Expected 2 classes but found {len(class_counts)}: {class_counts.index.tolist()}"

# Class weights: inverse of frequency - but capped to avoid extreme values
class_weights = total_samples / (len(class_counts) * class_counts)

# IMPORTANT: Don't use class weights with half precision - causes NaN
# Instead, we'll use None and rely on other techniques
class_weights_tensor = None  # Disabled for stability

print(f"\nClass weights calculation (for reference only):")
print(f"  Theoretical weights: {class_weights.values}")
print(f"  Class imbalance ratio (Non-PCL:PCL): {class_weights.values[0] / class_weights.values[1]:.2f}:1")
print(f"\n⚠ Class weights DISABLED to prevent NaN losses with mixed precision training")
print(f"  Using unweighted CrossEntropyLoss instead")


Class weights calculation (for reference only):
  Theoretical weights: [0.55538462 5.01388889]
  Class imbalance ratio (Non-PCL:PCL): 0.11:1

⚠ Class weights DISABLED to prevent NaN losses with mixed precision training
  Using unweighted CrossEntropyLoss instead


## 4. Create Dataset Class

In [5]:
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=192):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        # Validate text is not empty or just whitespace
        if not text or text.strip() == '':
            text = "[EMPTY]"  # Placeholder for empty text
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

## 5. Initialize Model and Tokenizer

In [6]:
import sentencepiece as spm

# Model configuration
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 192
BATCH_SIZE = 8  # Reduced from 32 to fit in GPU memory
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 8 * 4 = 32
LEARNING_RATE = 1e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1

print(f"Loading {MODEL_NAME}...")
tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
model = DebertaV2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type='single_label_classification'
)

# Clear GPU cache before moving model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.to(device)

print(f"Model loaded successfully!")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

Loading microsoft/deberta-v3-base...


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1698.19it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MI

Model loaded successfully!
Number of parameters: 184,423,682
Batch size: 8
Gradient accumulation steps: 4
Effective batch size: 32


## 6. Create Dataloaders

In [7]:
# Create datasets
train_dataset = PCLDataset(
    texts=train_df['text'].values,
    labels=train_df['label'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

dev_dataset = PCLDataset(
    texts=dev_df['text'].values,
    labels=dev_df['label'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Dev batches: {len(dev_loader)}")

Train batches: 948
Dev batches: 237


## 7. Setup Optimizer and Scheduler

In [8]:
# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Calculate total training steps (number of optimizer updates with gradient accumulation)
total_steps = (len(train_loader) // GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# Learning rate scheduler with warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

Total training steps: 1185
Warmup steps: 118


## 8. Training Functions

In [10]:
def train_epoch(model, dataloader, optimizer, scheduler, device, class_weights):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []
    nan_count = 0
    
    # Ensure class weights match model dtype (for mixed precision training)
    model_dtype = next(model.parameters()).dtype
    class_weights = class_weights.to(dtype=model_dtype)
    
    # Create loss function with class weights
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    
    progress_bar = tqdm(dataloader, desc='Training')
    optimizer.zero_grad()
    
    for idx, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        
        # Calculate loss
        loss = loss_fn(logits, labels)
        
        # Check for NaN or Inf
        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            print(f"\n⚠ Warning: Invalid loss detected at batch {idx}!")
            print(f"  Loss value: {loss.item()}")
            print(f"  Logits range: [{logits.min().item():.4f}, {logits.max().item():.4f}]")
            print(f"  Labels: {labels.cpu().numpy()}")
            # Skip this batch
            optimizer.zero_grad()
            continue
        
        # Backward pass
        loss.backward()
        
        # Only update weights every GRADIENT_ACCUMULATION_STEPS
        if (idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            # Gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Check for exploding gradients
            if grad_norm > 10.0:
                print(f"\n⚠ Warning: Large gradient norm detected: {grad_norm:.2f}")
            
            # Optimizer step
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item()
        
        # Get predictions
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        progress_bar.set_postfix({'loss': loss.item(), 'nan_count': nan_count})
    
    # Handle remaining gradients
    if (idx + 1) % GRADIENT_ACCUMULATION_STEPS != 0:
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = np.mean(np.array(predictions) == np.array(true_labels))
    f1 = f1_score(true_labels, predictions, average='binary')
    
    if nan_count > 0:
        print(f"\n⚠ Total NaN/Inf losses in epoch: {nan_count}")
    
    return avg_loss, accuracy, f1

In [9]:
def evaluate(model, dataloader, device, class_weights):
    """Evaluate the model"""
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    # Ensure class weights match model dtype (for mixed precision training)
    model_dtype = next(model.parameters()).dtype
    class_weights = class_weights.to(dtype=model_dtype)
    
    # Create loss function with class weights
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    
    progress_bar = tqdm(dataloader, desc='Evaluating')
    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            
            # Calculate loss
            loss = loss_fn(logits, labels)
            
            # Check for invalid loss
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"\n⚠ Warning: Invalid loss in evaluation! Skipping batch.")
                continue
            
            total_loss += loss.item()
            
            # Get predictions
            preds = torch.argmax(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = np.mean(np.array(predictions) == np.array(true_labels))
    
    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average='binary', zero_division=0
    )
    
    return avg_loss, accuracy, precision, recall, f1, predictions, true_labels

## 9. Training Loop with Early Stopping

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'train_f1': [],
    'dev_loss': [],
    'dev_acc': [],
    'dev_precision': [],
    'dev_recall': [],
    'dev_f1': []
}

# Early stopping parameters
best_f1 = 0
patience = 2
patience_counter = 0
best_model_path = 'best_deberta_model.pt'

print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 70)
    
    # Train
    train_loss, train_acc, train_f1 = train_epoch(
        model, train_loader, optimizer, scheduler, device,
        class_weights_tensor
    )
    
    # Evaluate
    dev_loss, dev_acc, dev_precision, dev_recall, dev_f1, _, _ = evaluate(
        model, dev_loader, device, class_weights_tensor
    )
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['dev_loss'].append(dev_loss)
    history['dev_acc'].append(dev_acc)
    history['dev_precision'].append(dev_precision)
    history['dev_recall'].append(dev_recall)
    history['dev_f1'].append(dev_f1)
    
    # Print metrics
    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Dev Loss: {dev_loss:.4f} | Dev Acc: {dev_acc:.4f}")
    print(f"Dev Precision: {dev_precision:.4f} | Dev Recall: {dev_recall:.4f} | Dev F1: {dev_f1:.4f}")
    
    # Early stopping based on F1 score
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        patience_counter = 0
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
            'history': history
        }, best_model_path)
        print(f"✓ New best model saved! (F1: {best_f1:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch + 1}")
            break

print("\n" + "="*70)
print("TRAINING COMPLETED")
print("="*70)
print(f"Best Dev F1 Score: {best_f1:.4f}")

## 10. Load Best Model and Final Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model loaded from epoch {checkpoint['epoch'] + 1}")
print(f"Best F1 score: {checkpoint['best_f1']:.4f}")

In [ ]:
# Final evaluation on dev set
dev_loss, dev_acc, dev_precision, dev_recall, dev_f1, predictions, true_labels = evaluate(
    model, dev_loader, device, class_weights_tensor
)

print("\n" + "="*70)
print("FINAL EVALUATION ON DEV SET")
print("="*70)
print(f"\nAccuracy: {dev_acc:.4f}")
print(f"Precision: {dev_precision:.4f}")
print(f"Recall: {dev_recall:.4f}")
print(f"F1 Score: {dev_f1:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(true_labels, predictions, target_names=['Non-PCL', 'PCL']))

## 11. Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(epochs_range, history['dev_loss'], 'r-', label='Dev Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[0, 0].set_title('Training and Validation Loss', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Accuracy
axes[0, 1].plot(epochs_range, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
axes[0, 1].plot(epochs_range, history['dev_acc'], 'r-', label='Dev Acc', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Accuracy', fontsize=11)
axes[0, 1].set_title('Training and Validation Accuracy', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# F1 Score
axes[1, 0].plot(epochs_range, history['train_f1'], 'b-', label='Train F1', linewidth=2)
axes[1, 0].plot(epochs_range, history['dev_f1'], 'r-', label='Dev F1', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('F1 Score', fontsize=11)
axes[1, 0].set_title('Training and Validation F1 Score', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Precision and Recall
axes[1, 1].plot(epochs_range, history['dev_precision'], 'g-', label='Precision', linewidth=2)
axes[1, 1].plot(epochs_range, history['dev_recall'], 'm-', label='Recall', linewidth=2)
axes[1, 1].plot(epochs_range, history['dev_f1'], 'r-', label='F1', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('Score', fontsize=11)
axes[1, 1].set_title('Dev Set: Precision, Recall, F1', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training history plot saved as 'training_history.png'")

## 12. Save Final Model

In [ ]:
# Save the full model and tokenizer
model.save_pretrained('./deberta_pcl_model')
tokenizer.save_pretrained('./deberta_pcl_model')

# Save configuration
import json
config = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'num_epochs': NUM_EPOCHS,
    'warmup_ratio': WARMUP_RATIO,
    'best_f1': float(best_f1),
    'best_epoch': checkpoint['epoch'] + 1,
    'class_weights': class_weights.values.tolist()
}

with open('./deberta_pcl_model/training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Model saved to './deberta_pcl_model'")
print("Configuration saved to './deberta_pcl_model/training_config.json'")

## 13. Model Summary

In [ ]:
print("\n" + "="*70)
print("MODEL SUMMARY")
print("="*70)
print(f"\nModel: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTraining Configuration:")
print(f"  - Max Sequence Length: {MAX_LENGTH}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Warmup Ratio: {WARMUP_RATIO}")
print(f"  - Class Weights: {class_weights.values}")
print(f"\nResults:")
print(f"  - Best Epoch: {checkpoint['epoch'] + 1}")
print(f"  - Dev F1 Score: {dev_f1:.4f}")
print(f"  - Dev Precision: {dev_precision:.4f}")
print(f"  - Dev Recall: {dev_recall:.4f}")
print(f"  - Dev Accuracy: {dev_acc:.4f}")
print("\n" + "="*70)